# Homework 3 — Training pix2pixHD on BBBC010

We use NVIDIA's [pix2pixHD](https://github.com/NVIDIA/pix2pixHD) on the BBBC010 data. The dataset has 80 train pairs (mask → brightfield) at
512×512. We train for 40 epochs and save **milestone checkpoints** at epochs 5, 10, 20, 40
so the evaluation notebook can compare quality over time.

## 1. Clone the original pix2pixHD repo

In [11]:
!git clone https://github.com/NVIDIA/pix2pixHD.git

Cloning into 'pix2pixHD'...
remote: Enumerating objects: 343, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 343 (delta 0), reused 0 (delta 0), pack-reused 340 (from 1)
Receiving objects: 100% (343/343), 55.68 MiB | 15.37 MiB/s, done.
Resolving deltas: 100% (156/156), done.


## 2. Apply the Python-3.11+ patches

The original repo targeted Python 3.5 / PyTorch 0.4. `apply_patches.py` (sitting next to
`pix2pixHD/`) makes three small changes

In [12]:
!python apply_patches.py

  patch pix2pixHD/train.py
  patch pix2pixHD/models/networks.py
  patch pix2pixHD/models/pix2pixHD_model.py
All patches applied.


## 3. Inspect data layout

After running `00_PrepData.ipynb` you should have:
```
datasets/bbbc010_pix2pixhd/
├── train_A/   # 80 binary masks (RGB, 512×512)
├── train_B/   # 80 brightfield images
├── test_A/    # 20 masks
└── test_B/    # 20 images
```
Because we use `--label_nc 0`, pix2pixHD treats the input as a raw image, not a semantic map.

## 4. Training command

Same flags as Unit 5. **Milestone checkpoints** (epochs 5/10/20/40) come from
`apply_patches.py` — they let the evaluation notebook show how generation quality evolves.

Expect ~1–2 hours on a single modern GPU. Adjust `--batchSize` to fit your memory.

In [13]:
!cd pix2pixHD && python train.py \
    --name bbbc010_512 \
    --dataroot ../datasets/bbbc010_pix2pixhd \
    --label_nc 0 \
    --no_instance \
    --loadSize 512 \
    --fineSize 512 \
    --batchSize 2 \
    --niter 40 \
    --niter_decay 0 \
    --save_epoch_freq 100 \
    --gpu_ids 0 \
    --checkpoints_dir ./checkpoints

Traceback (most recent call last):
  File "/workspaces/GAI4_course/HW_3/pix2pixHD/train.py", line 17, in <module>
    opt = TrainOptions().parse()
          ^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/GAI4_course/HW_3/pix2pixHD/options/base_options.py", line 80, in parse
    torch.cuda.set_device(self.opt.gpu_ids[0])
  File "/home/vscode/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py", line 529, in set_device
    torch._C._cuda_setDevice(device)
    ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: module 'torch._C' has no attribute '_cuda_setDevice'


## 5. Verify checkpoints

After training, you should see `5_net_*.pth`, `10_net_*.pth`, `20_net_*.pth`,
`40_net_*.pth` plus `latest_net_*.pth` in `pix2pixHD/checkpoints/bbbc010_512/`.

In [14]:
!ls -lthr pix2pixHD/checkpoints/bbbc010_512/

ls: cannot access 'pix2pixHD/checkpoints/bbbc010_512/': No such file or directory


## Conclusion

Mask-conditioned brightfield image generation: the model has to fill in
plausible worm textures inside the mask outline while leaving the background empty.
Next: `02_Evaluation.ipynb` quantifies how well it learned this over the 40-epoch run.